# 📡 Projet 5 — Modélisation Prédictive du Churn Client
## Télécommunications — Classification Supervisée

**Objectif :** Prédire quels clients risquent de se désabonner (churn) en utilisant plusieurs algorithmes de classification.

**Dataset :** Telco Customer Churn — [Kaggle](https://www.kaggle.com/datasets/blastchar/telco-customer-churn)

---
### Plan du Notebook
1. Chargement & Aperçu des Données
2. Nettoyage & Feature Engineering
3. Analyse Exploratoire (EDA)
4. Préparation des Données pour le ML
5. Modèles de Classification
   - Régression Logistique
   - K-Nearest Neighbors
   - Naive Bayes
   - Random Forest
   - Gradient Boosting
   - XGBoost
   - LightGBM
6. Comparaison des Modèles
7. Analyse des Erreurs
8. Insights & Recommandations

## 1. Chargement & Aperçu

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (confusion_matrix, roc_curve, auc, accuracy_score,
                              precision_score, recall_score, f1_score,
                              classification_report, roc_auc_score)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# Style
PAL = ['#7b9cff','#ef476f','#52b788','#f4a261','#64dfdf','#ffd166','#c77dff']
plt.rcParams.update({'figure.facecolor':'#131627','axes.facecolor':'#131627',
                     'axes.edgecolor':'#1e2540','axes.labelcolor':'#b0b8d8',
                     'xtick.color':'#b0b8d8','ytick.color':'#b0b8d8',
                     'text.color':'#b0b8d8','figure.dpi':110})

df = pd.read_csv('telco_churn.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
print('=== INFO DATASET ===')
print(df.dtypes)
print(f'\nValeurs manquantes:\n{df.isnull().sum()[df.isnull().sum()>0]}')
print(f'\nTaux de Churn: {df["Churn"].value_counts(normalize=True).round(3)}')

## 2. Nettoyage & Feature Engineering

In [ ]:
# Nettoyage
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)

# Feature Engineering
df['ChurnBin']         = (df['Churn'] == 'Yes').astype(int)
df['HasPartnerOrDep']  = ((df['Partner']=='Yes') | (df['Dependents']=='Yes')).astype(int)
df['HasStreaming']     = ((df['StreamingTV']=='Yes') | (df['StreamingMovies']=='Yes')).astype(int)
df['HasSecurity']      = ((df['OnlineSecurity']=='Yes') | (df['DeviceProtection']=='Yes')).astype(int)
df['AvgMonthlyRevenue'] = df['TotalCharges'] / (df['tenure'] + 1)
df['TenureGroup']      = pd.cut(df['tenure'], bins=[0,12,24,48,72],
                                 labels=['0-1 an','1-2 ans','2-4 ans','4-6 ans'])

print('Nouvelles features créées :', ['ChurnBin','HasPartnerOrDep','HasStreaming',
                                       'HasSecurity','AvgMonthlyRevenue','TenureGroup'])
df[['tenure','MonthlyCharges','TotalCharges','AvgMonthlyRevenue']].describe().round(2)

## 3. Analyse Exploratoire (EDA)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# Churn distribution
vals = df['Churn'].value_counts()
axes[0,0].pie(vals, labels=vals.index, colors=[PAL[1], PAL[0]],
              autopct='%1.1f%%', startangle=90)
axes[0,0].set_title('Distribution Churn')

# Churn par contrat
ct = df.groupby('Contract')['ChurnBin'].mean().sort_values(ascending=False)
axes[0,1].bar(ct.index, ct.values, color=PAL[:3])
axes[0,1].set_title('Taux de Churn par Contrat')
axes[0,1].set_ylabel('Taux de Churn')

# Tenure distribution
for label, c in [(0, PAL[0]), (1, PAL[1])]:
    axes[0,2].hist(df[df['ChurnBin']==label]['tenure'], bins=30,
                   alpha=0.7, color=c, label=['No Churn','Churn'][label])
axes[0,2].set_title('Distribution Ancienneté')
axes[0,2].legend()

# Monthly Charges
for label, c in [(0, PAL[0]), (1, PAL[1])]:
    axes[1,0].hist(df[df['ChurnBin']==label]['MonthlyCharges'], bins=30,
                   alpha=0.7, color=c, label=['No Churn','Churn'][label])
axes[1,0].set_title('Distribution Charges Mensuelles')
axes[1,0].legend()

# Churn par Internet
ct2 = df.groupby('InternetService')['ChurnBin'].mean()
axes[1,1].bar(ct2.index, ct2.values, color=PAL[:3])
axes[1,1].set_title('Churn par Service Internet')

# Scatter
for label, c, m in [(0,PAL[0],'o'), (1,PAL[1],'X')]:
    s = df[df['ChurnBin']==label].sample(200, random_state=42)
    axes[1,2].scatter(s['tenure'], s['MonthlyCharges'], c=c, alpha=0.4,
                      s=20, marker=m, label=['No Churn','Churn'][label])
axes[1,2].set_title('Tenure vs MonthlyCharges')
axes[1,2].legend()

plt.suptitle('EDA — Analyse du Churn Client', fontsize=14, fontweight='700', y=1.01)
plt.tight_layout()
plt.savefig('plots/01_eda_churn.png', bbox_inches='tight', dpi=120)
plt.show()

In [ ]:
# Heatmap corrélation
fig, ax = plt.subplots(figsize=(8, 5))
num_cols = ['tenure','MonthlyCharges','TotalCharges','SeniorCitizen',
            'HasPartnerOrDep','HasStreaming','HasSecurity','AvgMonthlyRevenue','ChurnBin']
corr = df[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', ax=ax,
            cmap='coolwarm', center=0, linewidths=0.5)
ax.set_title('Matrice de Corrélation')
plt.tight_layout()
plt.savefig('plots/02_correlation.png', bbox_inches='tight', dpi=120)
plt.show()

## 4. Préparation des Données

In [ ]:
CAT_COLS = ['gender','Partner','Dependents','PhoneService','MultipleLines',
            'InternetService','OnlineSecurity','OnlineBackup','DeviceProtection',
            'TechSupport','StreamingTV','StreamingMovies','Contract',
            'PaperlessBilling','PaymentMethod']
NUM_COLS = ['tenure','MonthlyCharges','TotalCharges','HasPartnerOrDep',
            'HasStreaming','HasSecurity','AvgMonthlyRevenue','SeniorCitizen']

dummies = pd.get_dummies(df[CAT_COLS], drop_first=True)
X = pd.concat([df[NUM_COLS], dummies], axis=1)
y = df['ChurnBin']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train: {X_train.shape} | Test: {X_test.shape}')
print(f'Features: {X.shape[1]}')
print(f'Churn train: {y_train.mean():.1%} | test: {y_test.mean():.1%}')

## 5. Entraînement des Modèles

In [ ]:
models = {
    'Régression Logistique': LogisticRegression(max_iter=1000, random_state=42),
    'K-Nearest Neighbors':   KNeighborsClassifier(n_neighbors=7),
    'Naive Bayes':           GaussianNB(),
    'Random Forest':         RandomForestClassifier(n_estimators=150, random_state=42, n_jobs=-1),
    'Gradient Boosting':     GradientBoostingClassifier(n_estimators=100, random_state=42),
    'XGBoost':               XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss'),
    'LightGBM':              LGBMClassifier(n_estimators=100, random_state=42, verbose=-1),
}

results = {}
for name, model in models.items():
    print(f'Training {name}...', end=' ')
    model.fit(X_train_sc, y_train)
    y_pred = model.predict(X_test_sc)
    y_prob = model.predict_proba(X_test_sc)[:,1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    cv = cross_val_score(model, X_train_sc, y_train, cv=StratifiedKFold(5), scoring='f1')
    results[name] = {
        'model':       model,
        'accuracy':    accuracy_score(y_test, y_pred),
        'precision':   precision_score(y_test, y_pred, zero_division=0),
        'recall':      recall_score(y_test, y_pred, zero_division=0),
        'f1':          f1_score(y_test, y_pred, zero_division=0),
        'roc_auc':     auc(fpr, tpr),
        'conf_matrix': confusion_matrix(y_test, y_pred),
        'fpr': fpr, 'tpr': tpr,
        'cv_f1': cv,
    }
    print(f'F1={results[name]["f1"]:.3f} | AUC={results[name]["roc_auc"]:.3f}')

print('\n✅ Tous les modèles entraînés!')

## 6. Comparaison des Modèles

In [ ]:
# Table de comparaison
rows = []
for name, r in results.items():
    rows.append({
        'Modèle':    name,
        'Accuracy':  round(r['accuracy'],4),
        'Precision': round(r['precision'],4),
        'Recall':    round(r['recall'],4),
        'F1-Score':  round(r['f1'],4),
        'ROC-AUC':   round(r['roc_auc'],4),
        'CV F1':     f"{r['cv_f1'].mean():.3f} ±{r['cv_f1'].std():.3f}"
    })

df_res = pd.DataFrame(rows).sort_values('F1-Score', ascending=False)
print('=== COMPARAISON DES MODÈLES ===')
df_res

In [ ]:
PAL = ['#7b9cff','#ef476f','#52b788','#f4a261','#64dfdf','#ffd166','#c77dff']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Courbes ROC
ax = axes[0]
for i, (name, r) in enumerate(results.items()):
    ax.plot(r['fpr'], r['tpr'], lw=2, color=PAL[i%len(PAL)],
            label=f'{name} (AUC={r["roc_auc"]:.3f})')
ax.plot([0,1],[0,1], 'k--', lw=1)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('Courbes ROC — Comparaison')
ax.legend(fontsize=7, loc='lower right')
ax.fill_between([0,1],[0,1], alpha=0.05, color='gray')

# Bar chart métriques
ax2 = axes[1]
metric_names = ['accuracy','precision','recall','f1','roc_auc']
x = np.arange(len(metric_names))
w = 0.8 / len(results)
for i, (name, r) in enumerate(results.items()):
    vals = [r[m] for m in metric_names]
    offset = (i - len(results)/2 + 0.5) * w
    ax2.bar(x + offset, vals, w, label=name, color=PAL[i%len(PAL)], alpha=0.85)
ax2.set_xticks(x)
ax2.set_xticklabels(['Accuracy','Precision','Recall','F1','AUC'], fontsize=9)
ax2.set_ylim(0.5, 1.0)
ax2.set_title('Comparaison des Métriques')
ax2.legend(fontsize=7, loc='lower right')

plt.suptitle('Évaluation des Modèles de Classification', fontsize=13, fontweight='700')
plt.tight_layout()
plt.savefig('plots/03_model_comparison.png', bbox_inches='tight', dpi=120)
plt.show()

In [ ]:
# Matrices de confusion — top 4 modèles
top_models = sorted(results.items(), key=lambda x: x[1]['f1'], reverse=True)[:4]
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, (name, r) in zip(axes, top_models):
    sns.heatmap(r['conf_matrix'], annot=True, fmt='d', ax=ax, cmap='Blues',
                xticklabels=['No Churn','Churn'],
                yticklabels=['No Churn','Churn'])
    ax.set_title(f'{name}\nF1={r["f1"]:.3f}', fontsize=9)
    ax.set_xlabel('Prédit'); ax.set_ylabel('Réel')

plt.suptitle('Matrices de Confusion — Top 4 Modèles', fontsize=12, fontweight='700')
plt.tight_layout()
plt.savefig('plots/04_confusion_matrices.png', bbox_inches='tight', dpi=120)
plt.show()

In [ ]:
# Feature Importance — Random Forest & XGBoost
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, model_name in zip(axes, ['Random Forest', 'XGBoost']):
    if model_name in results:
        m = results[model_name]['model']
        fi = pd.Series(m.feature_importances_, index=X.columns).sort_values(ascending=False).head(12)
        ax.barh(fi.index[::-1], fi.values[::-1], color=PAL[0], alpha=0.85)
        ax.set_title(f'Feature Importance — {model_name}')
        ax.set_xlabel('Importance')

plt.tight_layout()
plt.savefig('plots/05_feature_importance.png', bbox_inches='tight', dpi=120)
plt.show()

## 7. Insights & Recommandations

In [ ]:
best_name = max(results, key=lambda k: results[k]['f1'])
best = results[best_name]

print('=' * 60)
print('SYNTHÈSE — PROJET 5 : MODÉLISATION PRÉDICTIVE DU CHURN')
print('=' * 60)
print(f"\n🏆 Meilleur modèle : {best_name}")
print(f"   F1-Score  : {best['f1']:.3f}")
print(f"   ROC-AUC   : {best['roc_auc']:.3f}")
print(f"   Recall    : {best['recall']:.3f}")
print(f"   Precision : {best['precision']:.3f}")

print("\n📌 Facteurs Clés du Churn :")
print("   • Contrat mensuel → risque +3x vs contrat 2 ans")
print("   • Fibre optique → taux de churn plus élevé")
print("   • Ancienneté < 12 mois → période critique")
print("   • Absence de services additionnels (sécurité, support)")

print("\n💡 Recommandations Métier :")
print("   1. Cibler les clients mois-à-mois avec des offres d'upgrade")
print("   2. Programme d'onboarding intensif dans les 12 premiers mois")
print("   3. Offres groupées (internet + sécurité + support) pour réduire le churn")
print("   4. Alertes proactives pour les clients à risque (prob > 0.7)")
print("   5. Déploiement API Flask pour intégration CRM en temps réel")
print('=' * 60)